# Class weights against resampling

Logistic regression and gradient boosting, each with no weighting, class weights, random undersampling (10 legitimate per fraud) or SMOTE (frauds up to 10%). Scored by PR-AUC with 5 time-ordered folds on the training split, then on validation. Resampling sits inside an `imblearn` pipeline, so it only ever sees training-fold rows; `tests/test_compare.py` checks that with a spy sampler. Run by `make compare`.

In [1]:
import json
import os

import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

from fraud_pipeline import compare, config, ingest, split
from fraud_pipeline.schema import FEATURES, TARGET

os.chdir("..")
with open("reports/compare.json") as f:
    report = json.load(f)
rows = {
    name: {
        "CV mean": r["cv_pr_auc_mean"],
        "CV sd": r["cv_pr_auc_sd"],
        "validation PR-AUC": r["validation"]["pr_auc"],
        "validation recall @1/1000": r["validation"]["recall_at_1_per_1000"],
    }
    for name, r in report["results"].items()
}
pd.DataFrame(rows).T.sort_values("CV mean", ascending=False)

,CV mean,CV sd,validation PR-AUC,validation recall @1/1000
"boosting, class weights",0.7905,0.0759,0.8467,0.7321
"boosting, SMOTE",0.7789,0.0661,0.8481,0.7500
"boosting, undersampling",0.7625,0.0777,0.8377,0.7321
"logistic, SMOTE",0.7466,0.0704,0.8127,0.6071
"logistic, class weights",0.7253,0.0808,0.8394,0.7143
"logistic, undersampling",0.6771,0.1403,0.7454,0.6429
"logistic, no weighting",0.6349,0.0722,0.7815,0.5714
"boosting, no weighting",0.5886,0.1879,0.0021,0.0000


- **Weighting matters more than the model.** Unweighted logistic regression is the weakest of the logistic set (CV 0.635). Class weights lift it to 0.725, and SMOTE to 0.747.
- **Boosting with class weights is best on cross-validation** (0.791) and level with SMOTE on validation (0.847 against 0.848). Validation has only 56 frauds, so each one is worth about 1.8 points of recall, and differences under a few hundredths of PR-AUC are noise.
- **Undersampling throws away 99% of the legitimate data** and is never the best option.

## A dead end worth seeing: unweighted boosting

In [2]:
cfg, _ = config.load()
splits = split.by_time(ingest.load(cfg), cfg.split)
m = compare.candidates(cfg.seed)["boosting, no weighting"].fit(
    splits.train[FEATURES], splits.train[TARGET]
)
for name, part in [("train", splits.train), ("validation", splits.validation)]:
    p = m.predict_proba(part[FEATURES])[:, 1]
    print(
        f"{name:10s} mean score of frauds {p[part[TARGET] == 1].mean():.3g}, "
        f"of legitimate {p[part[TARGET] == 0].mean():.3g}"
    )

train      mean score of frauds 0.82, of legitimate 0.00125


validation mean score of frauds 4.02e-108, of legitimate 0.00355


Without weights or early stopping, 300 rounds of boosting memorise the 384 training frauds: their mean score is 0.82. On validation, the frauds it hasn't seen get about 10⁻¹⁰⁸, so it is confidently wrong, and PR-AUC falls to random (0.002). Its cross-validation mean (0.589) hid how badly it can fail. Class weights stop the collapse.

## How much does resampling before splitting inflate the score?

The classic mistake: SMOTE the whole training set, then cross-validate. Synthetic frauds are interpolated from real ones, so near-copies of every scored fraud sit in the training folds. Both versions use the same stratified folds and the same model; only where SMOTE runs differs.

In [3]:
x, y = splits.train[FEATURES], splits.train[TARGET]
folds = StratifiedKFold(5, shuffle=True, random_state=0)
smote = SMOTE(sampling_strategy=0.1, random_state=0)


def model():
    return LogisticRegression(class_weight=None, max_iter=5000)


right = Pipeline([("scale", StandardScaler()), ("smote", smote), ("model", model())])
right_scores = cross_val_score(right, x, y, cv=folds, scoring="average_precision")

x_res, y_res = smote.fit_resample(x, y)  # wrong: before splitting
wrong = Pipeline([("scale", StandardScaler()), ("model", model())])
wrong_scores = cross_val_score(wrong, x_res, y_res, cv=folds, scoring="average_precision")
pd.DataFrame(
    {"SMOTE inside folds": right_scores, "SMOTE before splitting": wrong_scores}
).describe().loc[["mean", "std"]].round(3)

,SMOTE inside folds,SMOTE before splitting
mean,0.761,0.983
std,0.055,0.001


**SMOTE before splitting reports 0.983 PR-AUC; done properly it's 0.761.** The 0.22 gap is pure leakage: the folds being scored share near-copies of their own frauds with the training folds. That's why resampling only lives inside the pipeline here.

## Decision

**Gradient boosting with class weights** goes forward to day 7's threshold. It has the best cross-validation PR-AUC, matches SMOTE on validation, and has one less moving part than resampling.